# 03. InstColorization 파이프라인

**Paper**: Instance-aware Image Colorization, CVPR 2020  
**Repo**: https://github.com/ericsujw/InstColorization

## 결과 요약 (보고서 기준)
* 원작자 Pre-train weight 의 Google Drive URL 이 만료되어 학습 진행 실패.
* 일부 checkpoint 만 우회 다운로드 가능했으나 완전한 weight set 부재로 학습/추론 실패.
* 본 노트북은 (1) repo clone + detectron2 셋업 (2) weight 다운로드 시도 (3) inference 실행 흐름을 정리합니다.
* 추가 참조: https://github.com/dayoungMM/Deepcolorization

## 1. 환경 셋업

원본 repo 가 PyTorch 1.5 + detectron2 0.1.2 를 요구해, 노트북 환경에 종속성이 큽니다.

In [ ]:
!pip install -q -U torch==1.5 torchvision==0.6 -f https://download.pytorch.org/whl/cu101/torch_stable.html
!pip install -q cython 'pyyaml==5.1' 'dominate==2.4.0' visdom
!pip install -q detectron2==0.1.2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.5/index.html

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/딥러닝
!git clone https://github.com/ericsujw/InstColorization.git InstColor_run || true
%cd InstColor_run

## 2. Pre-trained weight 다운로드 (실패 케이스)

원본 `download_model.sh` 는 만료된 Google Drive ID 를 사용합니다 (보고서 시점 기준). 우회 후보를 같이 시도합니다.

In [ ]:
!mkdir -p checkpoints
# 원본 스크립트
!sh scripts/download_model.sh || echo 'original URL failed'
# 백업: Berkeley siggraph_retrained 가중치 (일부)
!wget -q -O checkpoints/siggraph_retrained_latest_net_G.pth \
    http://colorization.eecs.berkeley.edu/siggraph/models/pytorch.pth || true
!ls checkpoints/

## 3. Inference 시도

weight 가 일부라도 다운된 경우만 통과합니다.

In [ ]:
import os
if os.path.exists('checkpoints/siggraph_retrained_latest_net_G.pth'):
    !sh scripts/test_mask.sh ../DATASET/test/gray_image Output_korean_food
else:
    print('weight 미확보 — 추론 스킵')

## 4. Fine-tune 시도 (참고)

weight 가 정상 확보되었다는 가정 하에 한국 음식 dataset 으로 fine-tune 흐름. 보고서 시점에는 weight 미확보로 진행 불가.

In [ ]:
# !sh scripts/prepare_train_box.sh ../DATASET/train
# !python train.py --stage fusion --name korean_food --sample_p 1.0 --niter 1 --niter_decay 0 \
#     --classification --load_model --fineSize 256 --batch_size 1 --display_ncols 0

## 메모

* `학습 진행 실패` 는 코드 자체 문제가 아니라 외부 weight 의존성 문제였습니다 — repo 자체는 동작합니다.
* 본 노트북은 흐름을 보존하고, weight 가 갱신되었을 때 즉시 재실행 가능한 형태로 정리했습니다.